In [2]:
import numpy as np
import pandas as pd
import requests
import duckdb

np.random.seed(123)

In [ ]:
# Récupération de l'URL des résultats de l'élection

BASE = "https://www.data.gouv.fr/api/1"
ds_elections = requests.get(f"{BASE}/datasets/donnees-des-elections-agregees/").json()

for r in ds_elections["resources"]:
    print(r["id"], "|", r["format"], "|", r["title"], "|", r.get("filesize"))

def get_url(title_part, fmt="parquet"):
    res = next(r for r in ds_elections["resources"]
               if title_part in r["title"] and r["format"].lower() == fmt)
    return res["url"]
url_general = get_url("Résultats généraux")
url_candidats = get_url("Résultats par candidat")    

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{url_candidats}')").show()

In [ ]:
# Valeurs de id_election disponibles pour 2026
print(duckdb.sql(f"""
    SELECT DISTINCT id_election
    FROM read_parquet('{url_candidats}')
    WHERE id_election LIKE '2026%'
""").df())

In [ ]:
t1_muni_26 = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url_candidats}')
    WHERE id_election == '2026_muni_t1'
""").df()
t1_muni_26.head()

In [ ]:
t2_muni_26 = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url_candidats}')
    WHERE id_election == '2026_muni_t2'
""").df()
t2_muni_26.head()

In [ ]:
# Récupération des données SIRENE

ds = requests.get(f"{BASE}/datasets/base-sirene-des-entreprises-et-de-leurs-etablissements-siren-siret/").json()
for r in ds["resources"]:
    print(r["id"], "|", r["format"], "|", r["title"], "|", r.get("filesize"))

In [ ]:
def get_url(title_part, fmt="parquet"):
    res = next(r for r in ds["resources"]
               if title_part.lower() in r["title"].lower() and r["format"].lower() == fmt)
    return res["url"]

url_etab = get_url("StockEtablissement")
url_ul = get_url("StockUniteLegale")

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{url_etab}')").show()

In [ ]:
# On récupère dans le répertoire les établissement dont l'activité principale est celle d'un débit de boissons (56.30Z)
bistrots_sirene = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url_etab}')
    WHERE activitePrincipaleEtablissement == '56.30Z'
""").df()
bistrots_sirene.head()

# Note : La variable du code commune s'appelle ici codeCommuneEtablissement.

In [ ]:
#Récupération des données Filosofi au niveau commune

def get_melodi(dataset, dimension, code, value_name, time_period, geo_level="COM"):

  url = f"https://api.insee.fr/melodi/data/{dataset}"

  params = {
    "GEO": geo_level,
    dimension: code,
    "TIME_PERIOD": time_period,
    "maxResult": 40000,
  }

  response = requests.get(url, params=params, timeout=60)
  response.raise_for_status()
  observations = response.json()["observations"]

  return pd.DataFrame(
    {
      "CODGEO": obs["dimensions"]["GEO"].split("-")[-1],
      value_name: obs["measures"]["OBS_VALUE_NIVEAU"].get("value"),
    }
    for obs in observations
  )

niveau_vie = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "MED_SL", "NIVVIE_MEDIAN", 2023)
taux_pauvrete = get_melodi("DS_FILOSOFI_CC", "FILOSOFI_MEASURE", "PR_MD60", "TAUX_PAUVRETE", 2023)
population = get_melodi("DS_POPULATIONS_HISTORIQUES", "POPREF_MEASURE", "PMUN", "POPULATION", 2023)

url_cog_2023 = "https://www.insee.fr/fr/statistiques/fichier/6800675/v_commune_2023.csv"
url_backup = "https://minio.lab.sspcloud.fr/lgaliana/data/python-ENSAE/cog_2023.csv"

try:
  cog_2023 = pd.read_csv(url_cog_2023)
except requests.exceptions.Timeout:
  cog_2023 = pd.read_csv(url_backup)

noms_communes = (
  cog_2023
  .loc[cog_2023["TYPECOM"] == "COM", ["COM", "LIBELLE"]]
  .rename(columns={"COM": "CODGEO", "LIBELLE": "LIBGEO"})
)

filosofi = (
  niveau_vie
  .merge(taux_pauvrete, on="CODGEO", how="outer")
  .merge(population, on="CODGEO", how="outer")
  .merge(noms_communes, on="CODGEO", how="left")
  [["CODGEO", "LIBGEO", "POPULATION", "NIVVIE_MEDIAN", "TAUX_PAUVRETE"]]
)

filosofi["POPULATION"] = filosofi["POPULATION"].astype(int)
filosofi["dep"] = filosofi["CODGEO"].str[:2]

KeyError: 'CODGEO'

In [ ]:
filosofi.head()